# `C16` — Multi-contract isolation

Scenario **C16** of the Track A certification ladder (card `OMIE-CERT-C16-1`, [`agile-plan.md`](../../docs/omiecid/agile-plan.md); canonical spec [`scenario-ladder.md`](../../docs/omiecid/scenario-ladder.md) §C16; plan [`scenario-ladder-plan.md`](../../docs/omiecid/scenario-ladder-plan.md) §C16 and §The capacity-validity precondition). Oracle kind: **exact-parametric (equality against the isolated runs)**.

**Terms used below** (defined once in `scenario-ladder.md`, "What eager, jitted and assertion mean here"): *eager* runs every JAX operation immediately (`jax.disable_jit()`), so a human can inspect each step; *jitted* traces and compiles the whole run (`jax.jit`), the mode training uses; the two must agree. An *assertion* records a failure and the notebook carries on: every check is evaluated, and the verdict comes at the end. A *draw* is one set of $n \ge 3$ contracts generated from one integer seed, each with its own tape. A *lane* is one contract's tape inside a stacked tape set, played by its own environment instance. Every pass/fail assertion is an **exact integer** comparison (order ids, trader ids, prices in ticks of 0.01 EUR/MWh, quantities in ticks of 0.1 MW, counts, flags); euro figures are shown, never asserted.

## 1. Purpose

**Purpose.** OMIE runs 96 quarter-hourly contracts a day and each has **its own** order book; there is
no matching across them. Since the environment carries tapes as one stacked array and slices per
contract, a leak between them is an entirely plausible indexing error — and a damaging one, because
it would manufacture trades that could not exist. The scenario takes the strong form: rather than
merely checking that no cross-contract trade appears, it runs each contract **in isolation** and
requires per-contract trade lists to be identical. That subsumes the weak check and also catches
state bleeding that produces no visible cross trade. Prices are deliberately overlapped so a leak
*would* match, and gates are staggered so one contract's closure cannot disturb another's.

*(Verbatim from `scenario-ladder.md` §C16.)*

**Certifies** (card): §6.4 no cross-contract matching; §2.1 the 96-contract structure; §2.2 *"Each contract has its own independent order book"*; §8.1, since $d$ is a contract property. **Known result** (card): per-contract trade lists identical to running each contract alone. **Pass** (card): per-contract exact equality. **Failure implicates** (card): book keying or tape-set slicing. Common to every scenario: eager and jitted agree; integer-tick assertions; the full battery runs; I8 capacity headroom holds on all five ceilings (a breach makes the draw **INVALID, not failed**: it is re-drawn and the discard rate is reported); I9 holds; every quoted passage matches its source.

**What C16 certifies, and what it does not.** C16 certifies **isolation**: a contract's outcome does not depend on which other contracts share its tape set, where it sits in the stack, how much padding the stack forces on it, or which contracts run beside it. It does **not** certify matching correctness. An equality between "contract alone" and "contract among others" is blind to any matching defect common to both arms: a wrong trade printed identically in both runs passes here. Matching correctness is the business of C02–C10 and C15. The battery runs on every isolated contract, but on random sessions it is a generic net with known holes (review V3: I5 checks the price-setter, not that the other leg's limit reaches the price). Iceberg contracts are included only so that the iceberg side channel (the hidden quantities held outside the visible book) is exercised for isolation. Nothing here certifies iceberg behaviour.

**What this notebook asserts, per valid draw** (section 5 states each part; section 7 evaluates and records it):

| | assertion | judged by |
|---|---|---|
| **P** | the dataset precondition: every contract id in $[0, 95]$ and distinct within the draw; one contract id per tape; prices, quantities, iceberg peaks and every ICEwPI slice price inside §2.3; the §13.6 sign rule; the slice margin $\lceil \mathit{total}/\mathit{peak} \rceil \le 64$; each tape's analytic capacity bounds inside the default envelope (≤ 80 resting per side, ≤ 80 trades, hop bound ≤ 160, ≤ 800 icebergs); order ids unique across the draw; timestamps positive, non-decreasing, and every contract's last order before its own gate $\mathit{GCT}(k)$ (§3.1); at least two distinct tape lengths and distinct last-order times (staggered); at least one **cross-contract crossable pair** (a resting order of one contract that an order of another contract would cross by price, so a leak *would* match); at least two contracts whose tape implies a trade | section 4 `precondition` |
| **R** | the **isolated engine run** of every contract (the battery's record builder: one fresh book keyed by the tape's contract id, that tape only): RB and I1–I9 **never `fail`** (`pass` and `not_applicable` are both acceptable; I8 `invalid` discards the draw; I9's "construction implies a trade" is section 5's `implies_trade`) | `run_battery` |
| **S** | **the strong form.** Every contract, run as one lane of a **stacked** multi-contract tape set through the environment, ends with exactly the outcome of its isolated engine run: the same trade list (every trade-row integer except the two timestamp columns: price, signed quantity, passive and aggressor order ids, passive and aggressor trader ids, row by row), the same resting book (per side, the multiset of price, quantity, order id, trader id), the same hidden remainder for every iceberg, the same `book_full` and `truncated` flags | section 5 `compare_outcomes` |
| **S′** | **the weak form, for diagnosis.** No trade of a lane names an order id that is not on that lane's own tape (§6.4); a foreign id is reported with the contract it came from | section 5 `foreign_oids` |
| **V** | §8.1: each contract's integer trade value $W_c = \sum_t q_t\,(p_t - p_0)$ is the same in the stacked lane and in the isolated run (implied by **S**, stated for §8.1), with $d = 0.25$ h for every contract | section 5 `trade_value_w` |

**On the first `ENV_ALONE_DRAWS` valid draws, additionally:** the draw's own stack (its $n$ distinct contract ids, in draw order, exactly the card's dataset) passes **S**; and every contract run **alone in the environment** (a one-tape set, padded only by its own tape) ends with the same outcome as its isolated engine run and as its lane in the draw's stack. **Eager == jitted** on the record builder for every contract of the first `EAGER_DRAWS` valid draws, and on the environment's stacked run of the first `ENV_EAGER_DRAWS` valid draws (whole final books, timestamps included). An expected eager comparison that did not happen is a **failed** `eager_jit` assertion.

**Aggregate assertions:** the section-6 wiring; the five documented ceilings; the configured contract count is 96 and $d = 0.25$ h $= 15$ min$/60$; the environment slicing precondition (`orders_per_step` ≤ the trailing pad rows); every requested draw ran (valid = requested); every valid draw ran in a pooled stacked run; the number of eager comparisons and environment-alone comparisons; contract ids 0 and 95 both exercised. **Reported, not asserted:** trade timestamps and the book's sequence counter, stacked vs isolated (finding F6); the step at which each lane's episode reports `done`, stacked vs alone (finding F8); the per-contract euro value $V_c$; capacity peaks; discard rate; wall clock.

**K2 findings** (recorded here; none blocks the expectation):

- **F1 — spec version.** The card names `market-spec.md` v2.12; the file is now **v2.13**. Every quote is checked against the current file at run time.
- **F2 — the §2.1 quote.** The card renders it as *"the quarter-hourly product … will be composed of 96 contracts"*: lower-case, with a bare ellipsis standing for an elision. The quote checker's only elision marker is `[…]`, so the source sentence is quoted **in full**, byte-copied. It contains its own `…`, which is source text, not an elision.
- **F3 — what "multi-contract" means in the environment.** `Session` is a single-contract environment: per its module docstring, a single agent trades **one** contract per episode. It carries a fixed tape set as one stacked `(n_tapes, N_orders_padded, 13)` int32 array and indexes it by the episode's `tape_idx`. Concurrent contracts are concurrent environment instances, `vmap`ped as training's `NUM_ENVS`. No single book holds two contracts, and the record builder refuses a tape with mixed contract ids. So §6.4's *"never matched against each other"* and §2.2's *"own independent order book"* are certified as follows: no lane's book is affected by any other tape in the set, by its position in the stack, by the stack's padding or by the lanes running beside it.
- **F4 — two isolated arms.** (i) The **engine alone**: the battery's record builder on the contract's own tape, for every contract of every draw. This is also the battery-judged run, so the battery and the stacked comparison look at the same thing. (ii) The **environment alone**: a one-tape set holding only that contract, with only its own padding, on the first `ENV_ALONE_DRAWS` draws. Every stacked lane must equal (i), and (ii) must equal both. The split exists because the environment's compile key includes the content of its tape set. Every distinct tape set is therefore a fresh compilation (10–30 s on a cold cache, per the card's runtime note), and running every contract alone in the environment would cost about 2 250 compilations.
- **F5 — pooled stacks.** For the same reason, valid draws are pooled `STACK_BATCH` at a time into one stacked tape set, with the lanes shuffled by a seeded permutation, so stack position never follows draw order. Within a draw the contract ids are distinct (the card's dataset). Across draws in one pool an id may repeat, as it does in a multi-date tape set (`build_fixed_tape_set` with one contract over several delivery dates). The draw-level stacks run on the first `ENV_ALONE_DRAWS` draws are exactly the card's dataset. The pooled stacks are larger and denser in overlapping prices, which makes a leak more likely to show.
- **F6 — what "identical trade list" covers.** It covers every integer of every trade row **except** the two timestamp columns (seconds, nanoseconds), plus the final book. Trade timestamps are **not** asserted because §5.2 assigns timestamps *"from a single global queue"*: in the real market, a contract's stamps can legitimately depend on activity in other contracts. Their agreement between arms, and that of the book's sequence counter, is reported. Resting entries are compared as a multiset per side, without timestamps, for the same reason. Time priority *among* them shows up in the trade list whenever it matters.
- **F7 — §8.1 and $d$.** $d$ comes from the product configuration, one static instance shared by every lane. Every OMIE contract is quarter-hourly (§2.1), so $d = 0.25$ h for all 96, and a leak of *another* contract's $d$ cannot be observed with this dataset. C16 asserts $d = 0.25$ h $=$ MTU$/60$, and that each contract's integer trade value $W_c$ is the same in both arms. $W_c$ is in units of $0.1\ \text{MW} \times 0.01\ \text{EUR/MWh}$; the euro value $V_c = W_c \cdot 0.1 \cdot 0.01 \cdot d$ is reported. $p_0$ is the tick code of 0 EUR/MWh.
- **F8 — gates in the environment.** The environment has no gate sweep. Its `done` is a clock that, by default (`session_clock = "span_scaled"`), scales each slice's latest timestamp against the **largest timestamp of the whole tape set**. A contract's `done` step and its clock `state.time` therefore depend on the *other* tapes in the set. This is not a matching leak, so it is outside C16's book claim. It is flagged for the orchestrator, because it is a cross-contract coupling of the session clock. *(REL-10 repeat, the maintainer's K7-U verdict of 2026-10-27, certification-log entry 17; corrected early-done ruling of 2026-10-26)* The release `Session` ignores every call after `done`. Under the default span ($S$ = the set's largest timestamp $M$) the clock rounds to the nearest minute, so `done` fires at the first slice whose largest timestamp is within half a scaled minute of $S$, possibly several slices before the lane's last real slice, and the rows after that slice are never replayed (section 9: seed 235, contract 95). Section 6 therefore builds every `Session`, stacked and environment-alone, with the full-replay span `tape_seconds_span` $= 2M$ over that `Session`'s own tape set (F10). The cross-contract coupling of the **default** clock described here remains true; it is stated in prose only, not measured. The reported first-`done` step per contract, stacked vs alone, on the draw-level runs is **measured under the full-replay span**, so the expected first-`done` step is −1 for every lane (no lane reaches `done`); it is reported, not asserted. Gates are staggered as §3.1 rules: each contract's last order lands 1–600 s before its own $\mathit{GCT}(k) = 60\lfloor k/4 \rfloor - 60$ min, measured in seconds from the gate opening (15:20 on D−1).
- **F9 — the slicing precondition.** Each environment step reads `orders_per_step` consecutive rows at the carried cursor. The environment docstring says the final short step's start is clamped, so it *"reads pad rows, never out of bounds"*. That holds only if a tape ends with at least `orders_per_step` − 1 pad rows. Otherwise the clamped last slice re-reads real rows. Every tape here ends with 8 pad rows (the generator's `pad_extra`), so `orders_per_step` is cycled over (4, 3, 8, 5), all ≤ 8, and the precondition is asserted. The cycling varies the slice boundaries, an incidental constant.
- **F10 — forcing a lane's tape.** `reset` samples `tape_idx` at random. The section-6 contract sets lane $i$'s `tape_idx` to $i$ after `reset`, submits `AgentOrder.none()` (no participant order), and calls `step_env` directly for all `steps_per_episode` steps. *(REL-10 repeat, the maintainer's K7-U verdict of 2026-10-27, certification-log entry 17; corrected early-done ruling of 2026-10-26)* The horizon is a **span** beyond every lane's clock: each `Session` (the stacked one and every environment-alone one) sets `tape_seconds_span` $S' = 2M$, with $M$ the largest real timestamp over that `Session`'s own tape set, and keeps the default `gct_minutes_from_got`. Every slice clock is then $\mathrm{round}(m_k\,\mathrm{gct}/S') \le \mathrm{round}(\mathrm{gct}/2) < \mathrm{gct}$, so no lane reaches `done` and no lane freezes: every tape is replayed in full within `steps_per_episode` steps. Raising `gct_minutes_from_got` does **not** help, and the earlier horizon `gct_minutes_from_got = 1_000_000` is withdrawn: the clock scales with gct, so `done` fires at the first slice with $m_k \ge S\,(1 - \tfrac{1}{2\,\mathrm{gct}})$. A larger gct only moves that threshold towards $S$, and a slice holding $S$ itself always reaches it (per the corrected ruling, the seed-235 lane of section 9 still reaches `done` at step 9 with gct $= 10^6$, stacked and alone). The research run's environment kept stepping after `done`, so it replayed every row whatever its clock said ("never stopping at `done`"); the release `Session` ignores calls after `done`, hence the explicit span.
- **F11 — the dataset.** No C16 generator mode exists. Each contract's tape is a `make_random_session` draw from the C15 regime grid (mixes × crossing probability × tie density × length). The notebook then applies only transformations that preserve the tape's structure: a price translation onto a common mid (so prices overlap across contracts), the contract id, a disjoint order-id block per contract, and a timestamp shift onto the contract's own gate. Translation leaves every analytic capacity bound and every within-tape crossing unchanged. The generator's capacity adaptation and slice margin are inherited, not re-implemented. Card deliverable (b) (a generator mode, if wanted) is K3's call. The composite draw's digest is recorded per draw.
- **F12 — non-vacuity.** `implies_trade(tape)` is a sound sufficient condition. It holds if some order $j$ that cannot be killed (NON, IOC or ICE) follows a resting-type order $i$ (NON or ICE) on the opposite side whose price it crosses. A tape carries no cancellation or expiry and stays inside the envelope, so either $i$ still rests visibly when $j$ arrives, or $i$ has already traded. The draw builder requires at least two trading contracts and at least one cross-contract crossable pair per draw, redrawing from the seed's own sub-streams.
- **F13 — book keying in the environment.** The environment builds its empty book from a fixed synthetic contract (`_synthetic_contract`, whose docstring says only the OPEN status matters). The record builder keys its book by the tape's contract id. If the engine applied §6.2 item 2's *"same contract"* test against the book's own id, lanes with id ≠ 0 would be refused, and C16 would fail on **S**, not pass vacuously, because the isolated run trades. K3 must not alter tape contract ids to make lanes trade.
- **F14 — record size.** One string per draw: seed, the composite digest, the slicing, and for each contract its id, length, trade count, a 10-hex digest of its isolated trade list, and the **S** / alone-environment letters. A failing contract is recorded in full: the differing trade rows (index, expected, observed), the missing and extra resting entries, the differing hidden remainders, any foreign order ids, the lane and the stack.


## 2. The rule, quoted

The governing passages, quoted **verbatim** (byte-copied) and checked against the document each names at run time (`notebooks.certification.cert_support.check_quotes`). A quote that has drifted is flagged **DRIFT** below and recorded as `quote_drift`: the notebook still runs to the end and ends `NOT CERTIFIED`. Text from the regulatory corpus is shown after them as **citations**, rendered but not machine-checked.

How the quotes ground the expectation of section 5:

- **One book per contract (§2.2, §6.4, §6.2 item 2).** A contract's outcome is a function of its own orders only. Hence **S**: equality with the run that sees nothing else. Hence **S′**: a trade naming another contract's order is the visible form of a leak.
- **96 quarter-hourly contracts (§2.1, §2.2).** Hence contract ids in $[0, 95]$, the configured count of 96, and the coverage of ids 0 and 95.
- **Rolling gates (§3.1).** Hence each contract's last order before its own $\mathit{GCT}(k)$, staggered (finding F8).
- **The global timestamp queue (§5.2).** Hence trade timestamps are reported, not asserted (finding F6).
- **$d$ (§2.3, §8.1).** Hence $d = 0.25$ h for every contract and the integer trade value **V** (finding F7).
- **The review's §8.3 correction (§3) and capacity requirements (§5).** Hence §8.1 rather than §8.3, the five-ceiling precondition and the reported discard rate.


In [ ]:
import time

T0 = time.time()   # the measured wall clock (results: wall_clock_s) starts here

from IPython.display import Markdown, display

from notebooks.certification.cert_support import (
    Certification, Citation, SpecQuote, check_quotes, render_quotes,
)

QUOTES = (
    # --- §2.2 / §6.4: one book per contract, no cross-contract matching -----------------------------------
    SpecQuote(
        "market-spec.md", "§2.2",
        "Each contract has its own independent order book.",
        note="A contract's outcome depends on its own orders only: the strong form S.",
    ),
    SpecQuote(
        "market-spec.md", "§6.4",
        "Orders on different contracts are never matched against each other.",
        note="No lane may trade against another lane's orders (S, and its visible form S').",
    ),
    SpecQuote(
        "market-spec.md", "§6.2 item 2",
        "Match is possible if: orders are opposite sides, same contract, and `sell_price ≤ buy_price`.",
        note="'Same contract' is part of matchability itself.",
    ),
    # --- §2.1 / §2.2: the 96-contract structure ------------------------------------------------------------
    SpecQuote(
        "market-spec.md", "§2.1",
        "The quarter-hourly product for the continuous intraday market will be composed of 96 contracts… "
        "each contract corresponds to each of the quarter-hourly periods of the day.",
        note="Finding F2: quoted in full; the ellipsis is source text. Contract ids are 0..95.",
    ),
    SpecQuote(
        "market-spec.md", "§2.1",
        "The v1 simulator must model **96 quarter-hourly (15-min) contracts per delivery day**.",
        note="The configured contract count (96) is asserted.",
    ),
    SpecQuote(
        "market-spec.md", "§2.2",
        "**96 contracts per delivery day**: one per 15-minute period (Q1–Q96, covering 00:00–24:00 CET).",
        note="Contract k (0-indexed) delivers 15k..15k+15 minutes after 00:00.",
    ),
    SpecQuote(
        "market-spec.md", "§2.2",
        "Contracts close on a **rolling basis**: approximately 4 contracts close per hour-long \"round\", each "
        "closing ~1 hour before its delivery period.",
        note="Gates are staggered across contracts.",
    ),
    # --- §3.1: the gates --------------------------------------------------------------------------------------
    SpecQuote(
        "market-spec.md", "§3.1",
        "With 0-indexed k counted from D 00:00, **GCT(k) = 60·⌊k/4⌋ − 60 min**.",
        note="Each contract's last order is placed 1-600 s before its own GCT(k) (finding F8).",
    ),
    SpecQuote(
        "market-spec.md", "§3.1",
        "All 96 contracts for delivery day D+1 open simultaneously after IDA1 completion at approximately "
        "15:20 CET on day D.",
        note="Tape timestamps are seconds from this gate opening (15:20 on D-1).",
    ),
    # --- §5.2: one global timestamp queue ---------------------------------------------------------------------
    SpecQuote(
        "market-spec.md", "§5.2",
        "All timestamps across orders and explicit capacity requests are assigned from a single global queue "
        "(unique, sequential).",
        note="Finding F6: a contract's stamps may depend on other contracts, so they are reported, not asserted.",
    ),
    # --- §2.3 / §8.1: d is a contract property ----------------------------------------------------------------
    SpecQuote(
        "market-spec.md", "§2.3",
        "**Trade value formula:** `V (EUR) = Q (MW) × p (EUR/MWh) × d (h)` where d = 0.25 h for 15-min contracts.",
        note="V: the per-contract trade value, in integer units, is the same in both arms.",
    ),
    SpecQuote(
        "market-spec.md", "§8.1",
        "`d`: delivery duration of the contract in hours — **for 15-min contracts: d = 0.25**",
        note="d belongs to the contract; every contract here is quarter-hourly (finding F7).",
    ),
    # --- bounds ---------------------------------------------------------------------------------------------------
    SpecQuote(
        "market-spec.md", "§2.3",
        "**Quantity bounds:** 0.1 MW (minimum) to 999 MW (maximum).",
        note="Precondition: every quantity within 1 to 9 990 ticks.",
    ),
    SpecQuote(
        "market-spec.md", "§2.3",
        "**Price bounds:** −9,999 EUR/MWh (minimum) to +9,999 EUR/MWh (maximum).",
        note="Precondition: every price, and every ICEwPI slice price, within 0 to 1 999 800 ticks.",
    ),
    SpecQuote(
        "market-spec.md", "§13.6 item 1",
        "**Sign rule (hard).** ICEwPI buy orders must have `PPD < 0`; ICEwPI sell orders must have `PPD > 0`.",
        note="Precondition.",
    ),
    # --- the domain review ------------------------------------------------------------------------------------------
    SpecQuote(
        "scenario-ladder-review.md", "§3",
        "**§8.3 was wrongly cited for per-contract settlement** in the multi-contract scenario. Corrected to "
        "§2.2 (*\"Each contract has its own independent order book\"*) and §8.1.",
        note="Why C16 cites §2.2 and §8.1, not §8.3.",
    ),
    SpecQuote(
        "scenario-ladder-review.md", "§5",
        "Added requirement: the precondition is checked **per draw and the discard rate reported**.",
        note="Discards are counted and reported.",
    ),
)

CITATIONS = (
    Citation(
        "OMIE Market Rules Annex 2 §9841–9857, as quoted in market-spec.md §2.1 (the corpus text itself is "
        "not read by the quote checker)",
        "The quarter-hourly product for the continuous intraday market will be composed of 96 contracts… each "
        "contract corresponds to each of the quarter-hourly periods of the day.",
    ),
    Citation(
        "docs/omiecid/scenario-ladder.md §C16, Dataset and Known result (not readable by the quote checker)",
        "`n_tapes ≥ 3` with distinct contract ids, overlapping price ranges, staggered gates; any iceberg on any "
        "tape within the slice margin $\\lceil \\mathit{total}/\\mathit{peak} \\rceil \\le 64$ […] Per-contract "
        "trade lists identical to running each contract alone",
    ),
    Citation(
        "docs/omiecid/scenario-ladder-plan.md §The capacity-validity precondition (five ceilings)",
        "The simulator has five hard ceilings, all consequences of fixed-size arrays and fixed loop bounds: "
        "100 resting orders per side, 100 trades per contract in the trade tape, 200 matching iterations, "
        "1024 concurrent icebergs, and 64 slices per iceberg (`MAX_SLICES_PER_REGEN`, `jax_constants.py` §B2).",
    ),
)

QUOTE_CHECKS = check_quotes(QUOTES)
display(Markdown(render_quotes(QUOTE_CHECKS, CITATIONS)))
print(f"{sum(c.ok for c in QUOTE_CHECKS)} of {len(QUOTE_CHECKS)} quotes match their source")


## 3. Configuration

One visible parameter cell. The headless gate overrides it by injecting a cell right after it (`execute_notebook(..., parameters={...})`); everything derived from it is computed in the next cell.

- **`DRAWS = 500`**: the card's default for random scenarios. Slot $j$ is drawn from seed `SEED0 + j`. A discarded slot (generation breach, failed precondition, a run that raised, or I8 `invalid`) is re-drawn from replacement seeds `SEED0 + DRAWS + j·REPLACEMENTS_PER_SLOT + r − 1`, $r = 1 \ldots$ `REPLACEMENTS_PER_SLOT`.
- **`N_CONTRACTS_MIN`, `N_CONTRACTS_MAX`**: contracts per draw ($n \ge 3$, card).
- **`STACK_BATCH`**: valid draws pooled into one stacked environment run (finding F5); one compilation each.
- **`ENV_ALONE_DRAWS`**: the first valid draws whose own stack runs, and whose every contract also runs alone in the environment (finding F4).
- **`ENV_EAGER_DRAWS`**, **`EAGER_DRAWS`**: eager-vs-jitted coverage on the environment's stacked run and on the record builder (an expected comparison that did not happen is a failure).
- **Derived, not parameters:** the C15 regime axes used per contract (mixes, crossing probability 0.2 / 0.4 / 0.6, tie density 0 / 0.4, lengths 16 / 24 / 32 / 48); `orders_per_step` cycled over (4, 3, 8, 5) per stacked run (finding F9); the five ceilings, read from the configuration and `jax_constants.py`, never typed in.


In [ ]:
CERT_ID = "C16"          # C00-C16 are the ladder; C90-C99 are reserved for the harness
DRAWS = 500              # valid draws requested (the card's default for random scenarios)
SEED0 = 0                # slot j draws from seed SEED0 + j
REPLACEMENTS_PER_SLOT = 4
N_CONTRACTS_MIN = 3      # contracts per draw (card: n_tapes >= 3)
N_CONTRACTS_MAX = 6
STACK_BATCH = 125        # valid draws pooled per stacked environment run (finding F5)
ENV_ALONE_DRAWS = 2      # first valid draws: own stack + every contract alone in the environment (F4)
ENV_EAGER_DRAWS = 1      # first valid draws: own stack eager vs jitted
EAGER_DRAWS = 2          # first valid draws: every contract's record eager vs jitted


In [ ]:
import hashlib
import math
import pathlib
from collections import namedtuple

import numpy as np

from omiecid_simulator.jaxob.jaxob_constants import TradesFeat
from omiecid_simulator.jaxomiecid import omie_tape_synth as synth
from omiecid_simulator.omiecid.encoding import decode_price, decode_qty, encode_price, encode_qty
from omiecid_simulator.omiecid.jax_constants import MAX_SLICES_PER_REGEN
from omiecid_simulator.omiecid.omie_config import OMIEMarketConfig, OMIEProductConfig
from omiecid_simulator.omiecid.order_types import ExecRestriction, Side

CONSTRUCTION = "C15.random_session"   # each contract's session; the C16 composite is built in section 4 (F11)
CONFIG = OMIEProductConfig()          # builds tapes and decodes for display; d and the MTU are read from it
DEFAULT_MARKET = OMIEMarketConfig()
ENVELOPE = synth.DEFAULT_ENVELOPE

BUY, SELL = int(Side.BUY), int(Side.SELL)
NON, IOC, FOK, ICE = (int(ExecRestriction.NON), int(ExecRestriction.IOC), int(ExecRestriction.FOK),
                      int(ExecRestriction.ICE))
TYPE_NAME = {NON: "NON", IOC: "IOC", FOK: "FOK", ICE: "ICE"}
C_CONTRACT, C_SIDE, C_PRICE, C_QTY, C_OID, C_AGENT, C_TS = (synth.COL_CONTRACT, synth.COL_SIDE, synth.COL_PRICE,
                                                             synth.COL_QTY, synth.COL_OID, synth.COL_AGENT,
                                                             synth.COL_TS)
C_EXEC, C_PEAK, C_PPD = synth.COL_EXEC, synth.COL_PEAK, synth.COL_PPD

PRICE_INT_MAX = encode_price(CONFIG.price_max, CONFIG)   # 1 999 800 (§2.3)
P0_INT = encode_price(0.0, CONFIG)                        # tick code of 0 EUR/MWh (999 900): p_0 of section 5
QTY_INT_MIN = encode_qty(CONFIG.qty_min, CONFIG)          # 1
QTY_INT_MAX = encode_qty(CONFIG.qty_max, CONFIG)          # 9 990
D_HOURS = float(CONFIG.delivery_duration_hours)           # d (§8.1)
NUM_CONTRACTS = int(DEFAULT_MARKET.num_contracts)         # 96 (§2.1)

# ---- the five ceilings, read from the configuration (never typed in) ---------------------------------------
CEILINGS = {
    "book_depth (resting orders per side)": int(DEFAULT_MARKET.book_depth),
    "nTrades (trade rows per contract)": int(DEFAULT_MARKET.nTrades),
    "max_match_iterations (per submission)": int(DEFAULT_MARKET.max_match_iterations),
    "max_icebergs (concurrent icebergs)": int(DEFAULT_MARKET.max_icebergs),
    "MAX_SLICES_PER_REGEN (slices per iceberg)": int(MAX_SLICES_PER_REGEN),
}
DOCUMENTED_CEILINGS = [100, 100, 200, 1024, 64]
SLICE_MARGIN = 64

# ---- trade-row columns (omiecid_simulator.jaxob.jaxob_constants.TradesFeat) ---------------------------------
T_P, T_Q, T_PASS_OID, T_AGRS_OID = (TradesFeat.P.value, TradesFeat.Q.value, TradesFeat.PASS_OID.value,
                                    TradesFeat.AGRS_OID.value)
T_SEC, T_NSEC, T_PASS_TID, T_AGRS_TID = (TradesFeat.SEC.value, TradesFeat.NSEC.value, TradesFeat.PASS_TID.value,
                                         TradesFeat.AGRS_TID.value)
TRADE_KEY_COLS = (T_P, T_Q, T_PASS_OID, T_AGRS_OID, T_PASS_TID, T_AGRS_TID)   # asserted (finding F6)
TRADE_KEY_NAMES = ("price_int", "qty_signed", "passive_oid", "aggressor_oid", "passive_tid", "aggressor_tid")

# ---- per-contract session axes (the C15 regime grid, finding F11) -----------------------------------------
MIXES = tuple(synth.FUZZ_TYPE_MIXES)          # non, non_ioc, non_ioc_fok, non_ioc_fok_ice, all_icewpi
CROSS_PROBS = (0.2, 0.4, 0.6)
TIE_DENSITIES = (0.0, 0.4)
LENGTHS = (16, 24, 32, 48)
MAX_BUILD_ATTEMPTS = 16      # sub-stream redraws per seed until the draw-level requirements hold (F12)
OID_BLOCK = 1000             # each contract's order ids live in their own block of 1 000 (> 48 orders)
OID_BLOCKS = 20_000          # blocks 1 .. 19 999: ids stay far below the agent id base 2e9
MID_EUR_RANGE = (40.0, 80.0) # the common mid of a draw, EUR/MWh
GATE_JITTER_S = (1, 600)     # last order lands 1-600 s before the contract's gate (finding F8)

# ---- the environment's slicing (finding F9) ----------------------------------------------------------------
PAD_EXTRA = 8                               # pad rows every tape ends with (TapeBuilder.build / stack_tapes)
ORDERS_PER_STEP_CYCLE = (4, 3, 8, 5)        # per stacked run; each <= PAD_EXTRA

# ---- the session frame (§3.1): timestamps are seconds from the gate opening, 15:20 on D-1 -----------------
GOT_MIN_FROM_D0 = -(24 * 60 - (15 * 60 + 20))   # -520 min


def gct_min_from_d0(k):
    """§3.1 correction: GCT(k) = 60 * floor(k / 4) - 60 min, from D 00:00."""
    return 60 * (int(k) // 4) - 60


def gate_seconds(k):
    """Contract k's gate in tape seconds (from the gate opening)."""
    return 60 * (gct_min_from_d0(k) - GOT_MIN_FROM_D0)


def period_label(k):
    return f"{(15 * int(k)) // 60:02d}:{(15 * int(k)) % 60:02d}"


def slot_seed(j, r=0):
    return int(SEED0) + j if r == 0 else int(SEED0) + int(DRAWS) + j * int(REPLACEMENTS_PER_SLOT) + (r - 1)


_own = sorted(pathlib.Path.cwd().glob(f"{CERT_ID}_*.ipynb"))
NOTEBOOK = _own[0].name if len(_own) == 1 else "C16_multi_contract_isolation.ipynb"

print(f"{CERT_ID}: {DRAWS} draws from seed {SEED0}, {N_CONTRACTS_MIN}-{N_CONTRACTS_MAX} contracts each "
      f"(up to {REPLACEMENTS_PER_SLOT} replacements per slot)")
print(f"stacks: {STACK_BATCH} draws per pooled stack, orders_per_step cycle {ORDERS_PER_STEP_CYCLE}, "
      f"{PAD_EXTRA} trailing pad rows")
print(f"environment-alone draws {ENV_ALONE_DRAWS}, environment eager draws {ENV_EAGER_DRAWS}, "
      f"record eager draws {EAGER_DRAWS}")
print(f"contracts configured: {NUM_CONTRACTS}; d = {D_HOURS} h; MTU = {CONFIG.mtu_minutes} min; p_0 = {P0_INT}")
print("the five ceilings (configuration):")
for _k, _v in CEILINGS.items():
    print(f"  {_k}: {_v}")
print(f"capacity envelope: {ENVELOPE}; slice margin <= {SLICE_MARGIN}")
print("gates (seconds from 15:20 on D-1): " + ", ".join(f"k={k} {period_label(k)} -> {gate_seconds(k)}"
                                                       for k in (0, 3, 4, 51, 95)))


## 4. Dataset

A draw is $n \in [3, 6]$ contracts with distinct ids $k \in [0, 95]$. Contract $c$'s tape is built from one sub-stream of the draw's `PCG64(seed)` stream:

1. a C15 session, `synth.make_random_session(sub_seed, regime)`, with the regime drawn from the grid (mix, crossing probability, tie density, length). The generator keeps it inside the capacity envelope and the slice margin by construction;
2. a **price translation** onto the draw's common mid (40–80 EUR/MWh): the session's price range is centred on it, so every contract's bids and asks straddle the same prices, *overlapping price ranges* (card);
3. the **contract id** $k$ on every real row;
4. a disjoint **order-id block** per contract (ids stay unique across the draw, as OMIE order ids are);
5. a **timestamp shift**: the session's last order lands 1–600 s before its own gate $\mathit{GCT}(k)$, measured in seconds from the gate opening, so the gates are *staggered* (card; §3.1).

Each transformation is a translation or a relabelling, so within-tape crossings and the analytic capacity bounds are unchanged. Draw-level requirements (at least one cross-contract crossable pair, at least two contracts that imply a trade, at least two distinct lengths) are met by redrawing from the seed's own sub-streams (`params["attempts"]`). The first slot's draw is forced to include contracts 0 and 95, the two ends of the day. Below: the first draw, in readable units beside the raw integers, and its price and timing shape.


In [ ]:
from IPython.display import Markdown, display

ContractTape = namedtuple("ContractTape", "cid tape mix cross tie n_orders sub_seed shift oid_base ts_shift gate_s "
                                          "jitter ice_oids")
C16Draw = namedtuple("C16Draw", "seed contracts params combined digest")


class DrawBuildFailure(ValueError):
    """The draw-level requirements could not be met in MAX_BUILD_ATTEMPTS sub-stream attempts."""


def real_rows(tape):
    return tape[tape[:, C_OID] >= 1]


def crossable(a, b):
    """Opposite sides and the buy limit reaches the sell limit (base prices, integer ticks)."""
    if int(a[C_SIDE]) == int(b[C_SIDE]):
        return False
    buy, sell = (a, b) if int(a[C_SIDE]) == BUY else (b, a)
    return int(buy[C_PRICE]) >= int(sell[C_PRICE])


def implies_trade(tape):
    """Finding F12: some NON/IOC/ICE order j crosses an earlier resting-type (NON/ICE) order i."""
    rows = real_rows(tape)
    for j in range(len(rows)):
        if int(rows[j][C_EXEC]) == FOK:
            continue
        for i in range(j):
            if int(rows[i][C_EXEC]) in (NON, ICE) and crossable(rows[i], rows[j]):
                return True
    return False


def leak_pairs(contracts):
    """Cross-contract crossable pairs: a resting-type order of one contract that an order of another crosses."""
    n = 0
    for a in contracts:
        ra = [r for r in real_rows(a.tape) if int(r[C_EXEC]) in (NON, ICE)]
        for b in contracts:
            if b.cid == a.cid:
                continue
            n += sum(1 for x in ra for y in real_rows(b.tape) if crossable(x, y))
    return n


def make_draw(seed):
    rng = np.random.Generator(np.random.PCG64(int(seed)))
    for attempt in range(MAX_BUILD_ATTEMPTS):
        n = int(rng.integers(N_CONTRACTS_MIN, N_CONTRACTS_MAX + 1))
        ids = [int(x) for x in rng.choice(NUM_CONTRACTS, size=n, replace=False)]
        if int(seed) == int(SEED0):            # the first slot exercises both ends of the day
            for forced, pos in ((0, 0), (NUM_CONTRACTS - 1, 1)):
                if forced not in ids:
                    ids[pos] = forced
        mid_eur = round(float(rng.uniform(*MID_EUR_RANGE)), 2)
        mid_int = int(encode_price(mid_eur, CONFIG))
        lengths = [int(x) for x in rng.choice(LENGTHS, size=n)]
        if len(set(lengths)) == 1:
            lengths[-1] = LENGTHS[(LENGTHS.index(lengths[-1]) + 1) % len(LENGTHS)]
        blocks = [int(x) for x in rng.choice(np.arange(1, OID_BLOCKS), size=n, replace=False)]
        contracts = []
        for i, k in enumerate(ids):
            mix = MIXES[int(rng.integers(0, len(MIXES)))]
            cp = CROSS_PROBS[int(rng.integers(0, len(CROSS_PROBS)))]
            td = TIE_DENSITIES[int(rng.integers(0, len(TIE_DENSITIES)))]
            regime = synth.fuzz_grid(mixes=(mix,), cross_probs=(cp,), tie_densities=(td,), lengths=(lengths[i],))[0]
            sub_seed = int(rng.integers(0, 2**31 - 1))
            jitter = int(rng.integers(GATE_JITTER_S[0], GATE_JITTER_S[1] + 1))
            session = synth.make_random_session(sub_seed, regime)   # may raise synth.CapacityBreach
            tape = np.array(session.tape, dtype=np.int32, copy=True)
            real = tape[:, C_OID] >= 1
            prices = tape[real, C_PRICE].astype(np.int64)
            shift = mid_int - int((int(prices.min()) + int(prices.max())) // 2)
            tape[real, C_PRICE] = (prices + shift).astype(np.int32)
            tape[real, C_CONTRACT] = k
            oid_base = blocks[i] * OID_BLOCK
            tape[real, C_OID] = tape[real, C_OID] + oid_base
            ts = tape[real, C_TS].astype(np.int64)
            ts_shift = gate_seconds(k) - jitter - int(ts.max())
            tape[real, C_TS] = (ts + ts_shift).astype(np.int32)
            tape = np.ascontiguousarray(tape, dtype=np.int32)
            ice = tuple(int(r[C_OID]) for r in real_rows(tape) if int(r[C_EXEC]) == ICE)
            contracts.append(ContractTape(k, tape, mix, cp, td, int(lengths[i]), sub_seed, int(shift), oid_base,
                                          int(ts_shift), gate_seconds(k), jitter, ice))
        n_trading = sum(implies_trade(c.tape) for c in contracts)
        n_leak = leak_pairs(contracts)
        if n_trading >= 2 and n_leak >= 1:
            combined = synth.pad_to(np.concatenate([real_rows(c.tape) for c in contracts], axis=0),
                                    sum(len(real_rows(c.tape)) for c in contracts) + PAD_EXTRA)
            params = {
                "n_contracts": n, "contract_ids": ids, "mid_eur": mid_eur, "mid_int": mid_int,
                "attempts": attempt + 1, "leak_pairs": int(n_leak), "trading_contracts": int(n_trading),
                "contracts": [{"cid": c.cid, "period": period_label(c.cid), "mix": c.mix, "cross_prob": c.cross,
                               "tie_density": c.tie, "n_orders": c.n_orders, "sub_seed": c.sub_seed,
                               "price_shift_int": c.shift, "oid_base": c.oid_base, "ts_shift": c.ts_shift,
                               "gate_s": c.gate_s, "jitter_s": c.jitter, "implies_trade": implies_trade(c.tape),
                               "icebergs": len(c.ice_oids)} for c in contracts],
            }
            return C16Draw(int(seed), tuple(contracts), params, combined, synth.tape_digest(combined))
    raise DrawBuildFailure(f"seed {seed}: no draw with >= 2 trading contracts and a cross-contract crossable "
                           f"pair in {MAX_BUILD_ATTEMPTS} attempts")


def precondition(draw):
    """Every problem with a draw (empty list = the dataset precondition holds)."""
    probs = []
    cs = draw.contracts
    ids = [c.cid for c in cs]
    if len(cs) < 3:
        probs.append(f"only {len(cs)} contracts (card: n_tapes >= 3)")
    if len(set(ids)) != len(ids):
        probs.append(f"contract ids not distinct: {ids}")
    if len({c.n_orders for c in cs}) < 2:
        probs.append("every tape has the same length")
    lasts = []
    all_oids = []
    for c in cs:
        t = c.tape
        tag = f"contract {c.cid}"
        if not (isinstance(t, np.ndarray) and t.ndim == 2 and t.shape[1] == synth.N_COLS and t.dtype == np.int32):
            probs.append(f"{tag}: tape is not an (N, 13) int32 array")
            continue
        if not 0 <= c.cid < NUM_CONTRACTS:
            probs.append(f"{tag}: id outside [0, {NUM_CONTRACTS - 1}] (§2.1)")
        rows = real_rows(t)
        if len(rows) == 0:
            probs.append(f"{tag}: no real rows")
            continue
        if set(int(x) for x in rows[:, C_CONTRACT]) != {c.cid}:
            probs.append(f"{tag}: real rows carry contract ids {sorted(set(int(x) for x in rows[:, C_CONTRACT]))}")
        if int(np.sum(t[-PAD_EXTRA:, C_OID] >= 1)) != 0:
            probs.append(f"{tag}: the last {PAD_EXTRA} rows are not all pad rows (finding F9)")
        ts = rows[:, C_TS].astype(np.int64)
        if ts.min() < 1 or np.any(np.diff(ts) < 0):
            probs.append(f"{tag}: timestamps not positive and non-decreasing")
        if int(ts.max()) >= gate_seconds(c.cid):
            probs.append(f"{tag}: last order at {int(ts.max())} s is not before its gate {gate_seconds(c.cid)} s (§3.1)")
        lasts.append(int(ts.max()))
        all_oids.extend(int(o) for o in rows[:, C_OID])
        for r in rows:
            p, q, ex, pk, pp, side = (int(r[C_PRICE]), int(r[C_QTY]), int(r[C_EXEC]), int(r[C_PEAK]), int(r[C_PPD]),
                                      int(r[C_SIDE]))
            oid = int(r[C_OID])
            if not 0 <= p <= PRICE_INT_MAX:
                probs.append(f"{tag} oid {oid}: price {p} outside [0, {PRICE_INT_MAX}] (§2.3)")
            if not QTY_INT_MIN <= q <= QTY_INT_MAX:
                probs.append(f"{tag} oid {oid}: quantity {q} outside [{QTY_INT_MIN}, {QTY_INT_MAX}] (§2.3)")
            if ex == ICE:
                if not 0 < pk <= q:
                    probs.append(f"{tag} oid {oid}: iceberg peak {pk} not in (0, {q}]")
                    continue
                slices = -(-q // pk)
                if slices > SLICE_MARGIN:
                    probs.append(f"{tag} oid {oid}: {slices} slices > {SLICE_MARGIN}")
                if (side == BUY and pp > 0) or (side == SELL and pp < 0):
                    probs.append(f"{tag} oid {oid}: ICEwPI sign rule (§13.6 item 1)")
                last = p + (slices - 1) * pp
                if not (0 <= last <= PRICE_INT_MAX):
                    probs.append(f"{tag} oid {oid}: ICEwPI slice price {last} outside §2.3")
            elif pk != 0 or pp != 0:
                probs.append(f"{tag} oid {oid}: non-iceberg row with peak {pk} / ppd {pp}")
        try:
            synth.check_envelope(ENVELOPE, **synth.tape_capacity_bounds(t))
        except synth.CapacityBreach as exc:
            probs.append(f"{tag}: capacity bounds outside the envelope ({exc})")
    if len(set(all_oids)) != len(all_oids):
        probs.append("order ids repeat across the draw")
    if len(set(lasts)) != len(lasts):
        probs.append(f"last-order times not staggered: {lasts}")
    if leak_pairs(cs) < 1:
        probs.append("no cross-contract crossable pair: a leak could not match")
    if sum(implies_trade(c.tape) for c in cs) < 2:
        probs.append("fewer than two contracts imply a trade")
    return probs


FIRST = make_draw(slot_seed(0))
rows_md = ["| contract | period | gate GCT(k) | mix | cross | tie | orders | icebergs | price range (EUR/MWh) | "
           "price_int range | oid range | last order (s) | gate (s) | implies a trade |",
           "|---|---|---|---|---|---|---|---|---|---|---|---|---|---|"]
for c in FIRST.contracts:
    r = real_rows(c.tape)
    gm = gct_min_from_d0(c.cid)
    gate_txt = f"{'D-1 ' if gm < 0 else 'D '}{(gm % 1440) // 60:02d}:{gm % 60:02d}"
    rows_md.append(
        f"| {c.cid} | {period_label(c.cid)} | {gate_txt} | {c.mix} | {c.cross} | {c.tie} | {len(r)} | {len(c.ice_oids)} | "
        f"{decode_price(int(r[:, C_PRICE].min()), CONFIG):.2f} – {decode_price(int(r[:, C_PRICE].max()), CONFIG):.2f} | "
        f"{int(r[:, C_PRICE].min())} – {int(r[:, C_PRICE].max())} | {int(r[:, C_OID].min())} – {int(r[:, C_OID].max())} | "
        f"{int(r[:, C_TS].max())} | {c.gate_s} | {implies_trade(c.tape)} |")
display(Markdown(f"**Seed {FIRST.seed}** (composite digest `{FIRST.digest[:16]}…`, {FIRST.params['attempts']} "
                 f"attempt(s), {FIRST.params['leak_pairs']} cross-contract crossable pairs, common mid "
                 f"{FIRST.params['mid_eur']:.2f} EUR/MWh)\n\n" + "\n".join(rows_md)))

_c = FIRST.contracts[0]
rows_md = ["| oid | side | type | price (EUR/MWh) | qty (MW) | price_int | qty_int | peak_int | ppd_int | agent | ts (s) |",
           "|---|---|---|---|---|---|---|---|---|---|---|"]
for r in real_rows(_c.tape)[:12]:
    rows_md.append(
        f"| {int(r[C_OID])} | {'BUY' if int(r[C_SIDE]) == BUY else 'SELL'} | {TYPE_NAME[int(r[C_EXEC])]} | "
        f"{decode_price(int(r[C_PRICE]), CONFIG):.2f} | {decode_qty(int(r[C_QTY]), CONFIG):.1f} | {int(r[C_PRICE])} | "
        f"{int(r[C_QTY])} | {int(r[C_PEAK])} | {int(r[C_PPD])} | {int(r[C_AGENT])} | {int(r[C_TS])} |")
display(Markdown(f"First 12 rows of contract {_c.cid} ({period_label(_c.cid)}):\n\n" + "\n".join(rows_md)))
print(f"precondition on seed {FIRST.seed}: {precondition(FIRST) or 'holds'}")


In [ ]:
import matplotlib.pyplot as plt

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 3.4))
for i, c in enumerate(FIRST.contracts):
    r = real_rows(c.tape)
    for side, mk in ((BUY, "^"), (SELL, "v")):
        sel = r[r[:, C_SIDE] == side]
        ax1.scatter(np.full(len(sel), i) + (0.12 if side == SELL else -0.12),
                    [decode_price(int(p), CONFIG) for p in sel[:, C_PRICE]], marker=mk, s=14,
                    color=f"C{i}", label=f"{c.cid} {'BUY' if side == BUY else 'SELL'}")
    ax2.plot([int(r[:, C_TS].min()) / 3600, int(r[:, C_TS].max()) / 3600], [i, i], color=f"C{i}", lw=4)
    ax2.plot([c.gate_s / 3600], [i], marker="|", markersize=16, color="black")
ax1.axhline(FIRST.params["mid_eur"], color="grey", lw=0.8, ls="--")
ax1.set_xticks(range(len(FIRST.contracts)), [f"k={c.cid}" for c in FIRST.contracts])
ax1.set_ylabel("limit price (EUR/MWh)")
ax1.set_title("overlapping price ranges (BUY ^, SELL v)")
ax2.set_yticks(range(len(FIRST.contracts)), [f"k={c.cid} ({period_label(c.cid)})" for c in FIRST.contracts])
ax2.set_xlabel("hours from the gate opening (15:20 on D-1)")
ax2.set_title("order activity (bar) and gate GCT(k) (|): staggered")
plt.tight_layout()
plt.show()


## 5. Known result

For every contract $c$ of a draw, let $T_c$ be its tape and $\mathcal{O}(\cdot)$ the **outcome** of a run, read from the final book:

$$\mathcal{O} = \big(\ \text{trade rows } (p,\ q^{\pm},\ o_{\text{pass}},\ o_{\text{aggr}},\ a_{\text{pass}},\ a_{\text{aggr}})\ \text{in row order};\ \ \text{resting bids};\ \ \text{resting asks};\ \ \text{hidden remainder per iceberg};\ \ (\text{book\_full},\ \text{truncated})\ \big),$$

with resting entries as multisets of (price, quantity, order id, trader id) and the timestamp columns left out (finding F6). Each contract has its own independent order book (§2.2), and orders on different contracts are never matched against each other (§6.4). So a contract's outcome is a function of $T_c$ **alone**:

$$\mathcal{O}\big(\text{lane of } T_c \text{ in any stack } \{T_c, T_{c'}, \ldots\}\big) \;=\; \mathcal{O}\big(\text{isolated engine run of } T_c\big) \;=\; \mathcal{O}\big(\text{environment run of } \{T_c\}\big).$$

The middle term is the expectation. It is an *exact-parametric* golden, in that the dataset is random and the answer is still known, but here the "parameter" is the contract's own tape, run where nothing else exists. Corollaries, stated so that a failure says which one broke:

- **S′ (weak form).** Every order id in a lane's trades is on $T_c$. A foreign id comes from another contract.
- **V (§8.1).** $W_c = \sum_t |q_t|\,(p_t - p_0)$ in integer units ($p_0$ = the tick code of 0 EUR/MWh) is identical in every arm. The euro value is $V_c = W_c \cdot 0.1 \cdot 0.01 \cdot d$ with $d = 0.25$ h, reported.
- **Non-vacuity.** If `implies_trade(T_c)`, the isolated run trades (I9), so equality with it is never satisfied by two empty runs.

Below, the functions section 7 applies, and the pre-run facts of the first draw: which contracts imply a trade, and the cross-contract crossable pairs a leak would turn into trades.


In [ ]:
def trade_rows(snap):
    """The engine's used trade rows (SEC != -1), in row order."""
    t = np.asarray(snap.trades)
    return t[t[:, T_SEC] != -1] if t.size else t.reshape(0, 8)


def outcome_of(snap, ice_oids):
    """(asserted outcome, reported auxiliaries) of one final book (finding F6)."""
    used = trade_rows(snap)
    out = {
        "trades": tuple(tuple(int(r[col]) for col in TRADE_KEY_COLS) for r in used),
        "bids": tuple(sorted((p, q, o, a) for (p, q, o, a, _s, _n) in snap.resting(BUY))),
        "asks": tuple(sorted((p, q, o, a) for (p, q, o, a, _s, _n) in snap.resting(SELL))),
        "hidden": tuple((int(o), int(snap.hidden_qty(o))) for o in sorted(ice_oids)),
        "flags": (bool(snap.book_full), bool(snap.truncated)),
    }
    aux = {"stamps": tuple((int(r[T_SEC]), int(r[T_NSEC])) for r in used),
           "ts_counter": int(snap.ts_counter), "n_active": int(snap.n_active)}
    return out, aux


def _multiset_diff(exp, obs):
    from collections import Counter
    e, o = Counter(exp), Counter(obs)
    return {"missing": sorted((e - o).elements()), "extra": sorted((o - e).elements())}


def compare_outcomes(exp, obs):
    """{field: difference} for every field that differs (empty dict = identical)."""
    d = {}
    if exp["trades"] != obs["trades"]:
        n = max(len(exp["trades"]), len(obs["trades"]))
        d["trades"] = {"n_expected": len(exp["trades"]), "n_observed": len(obs["trades"]),
                       "rows": [[i, list(exp["trades"][i]) if i < len(exp["trades"]) else None,
                                 list(obs["trades"][i]) if i < len(obs["trades"]) else None]
                                for i in range(n)
                                if (exp["trades"][i] if i < len(exp["trades"]) else None)
                                != (obs["trades"][i] if i < len(obs["trades"]) else None)]}
    for side in ("bids", "asks"):
        if exp[side] != obs[side]:
            d[side] = _multiset_diff(exp[side], obs[side])
    if exp["hidden"] != obs["hidden"]:
        e, o = dict(exp["hidden"]), dict(obs["hidden"])
        d["hidden"] = [[k, e.get(k), o.get(k)] for k in sorted(set(e) | set(o)) if e.get(k) != o.get(k)]
    if exp["flags"] != obs["flags"]:
        d["flags"] = {"expected": list(exp["flags"]), "observed": list(obs["flags"])}
    return d


def foreign_oids(out, own_oids, oid_owner):
    """S': order ids in the trades that are not on the lane's own tape -> {oid: owning contract or None}."""
    seen = {r[2] for r in out["trades"]} | {r[3] for r in out["trades"]}
    return {int(o): oid_owner.get(int(o)) for o in sorted(seen) if int(o) not in own_oids}


def trade_value_w(out):
    """V: W_c = sum |q| (p - p_0), integer units of 0.1 MW x 0.01 EUR/MWh."""
    return int(sum(abs(r[1]) * (r[0] - P0_INT) for r in out["trades"]))


def value_eur(w):
    return w * CONFIG.qty_tick * CONFIG.price_tick * D_HOURS


def trade_digest(out):
    return hashlib.blake2b(repr(out["trades"]).encode("ascii"), digest_size=5).hexdigest()


for c in FIRST.contracts:
    print(f"contract {c.cid:2d} ({period_label(c.cid)}): {c.n_orders} orders, implies a trade: {implies_trade(c.tape)}; "
          f"icebergs {len(c.ice_oids)}")
print(f"cross-contract crossable pairs in seed {FIRST.seed}: {leak_pairs(FIRST.contracts)} "
      f"(each would be a trade if the books leaked)")
print(f"expected for every contract c and every arm: O(lane) == O(isolated record); W_c identical; "
      f"d = {D_HOURS} h; p_0 = {P0_INT}")


## 6. Run

### The isolated engine run, and the environment's stacked multi-contract run (stub — to be wired at K3)

> **K2 stub.** This section is deliberately empty. The developer wires it at K3 and edits **nothing else** (independence rule 2): sections 1–5 and 7 are the expectation. A disagreement with the expectation is escalated as a `SCOPE-CHANGE` candidate with both readings, never resolved by editing sections 5 or 7.

Section 7 consumes exactly **two names** from this section. While either is undefined, section 7 records a failing `section 6 wired` check, runs no draw, and the notebook ends `NOT CERTIFIED`.

1. **`run_record(tape, *, jit) -> RunRecord`**: the isolated engine run, exactly
   `tests.omiecid.reference.omie_invariants.build_record(tape, capacity_bounds=synth.tape_capacity_bounds(tape), jit=jit)`,
   with no other argument: one fresh book of an OPEN contract keyed by the tape's contract id, the default `OMIEProductConfig()` and `OMIEMarketConfig()`, no cancellation, no gate expiry. `tape` is one contract's `(N, 13)` int32 tape (section 4), ending with 8 pad rows. `jit=False` runs eager, `jit=True` jitted. The function may raise.

2. **`run_stacked(tapes, *, orders_per_step, jit) -> tuple[(BookSnapshot, int), ...]`**: the environment's multi-contract path on a stacked tape set.
   - `tapes`: an `(n, N_padded, 13)` int32 array from `synth.stack_tapes` ($n \ge 1$; $n = 1$ is "the contract alone in the environment"). `orders_per_step` ≤ 8 (finding F9).
   - Build **one** `omiecid_simulator.omiecid.omie_session.Session({"tapes": tapes, "orders_per_step": orders_per_step, "tape_seconds_span": 2.0 * M})`, every other key at its default (default product and market configurations, `session_clock = "span_scaled"`, and the **default** `gct_minutes_from_got`: do not pass it). Here $M$ = `float(max(int(tapes[..., C_TS].max()), 1))`, the largest real timestamp over **this `Session`'s own tape set**, i.e. over the `tapes` argument of this call: for a pooled or draw-level stack, the largest timestamp over all $n$ lanes of that stack; for a contract alone ($n = 1$, `tapes = tape[None]`), the largest timestamp of that contract's own tape. Each call computes its own $M$; none is shared across calls. With $S' = 2M$ the exact no-`done` condition $\mathrm{round}(M\,\mathrm{gct}/S') < \mathrm{gct}$, i.e. $S' > M\,\mathrm{gct}/(\mathrm{gct}-\tfrac12)$, holds with margin (every slice clock stays at or below $\mathrm{round}(\mathrm{gct}/2)$, so float32 rounding cannot matter), and **no lane reaches `done` within `env.steps_per_episode` steps**. *(REL-10 repeat, the maintainer's K7-U verdict of 2026-10-27, certification-log entry 17; corrected early-done ruling of 2026-10-26; findings F8, F10. The earlier horizon `gct_minutes_from_got = 1_000_000` is withdrawn: it does not prevent an early `done`, section 9.)*
   - Run $n$ lanes **concurrently with `jax.vmap`** (training's `NUM_ENVS` path), lane $i$ playing tape $i$: `reset` (any key per lane), then set the returned state's `tape_idx` to $i$ (finding F10), then exactly `env.steps_per_episode` calls of `step_env` with **`AgentOrder.none()`** (no participant order). Use `step_env` directly, and do not stop at `done`.
   - Return, per lane and in stack order, (a) the final book as a `tests.omiecid.reference.omie_invariants.BookSnapshot`, read from the lane's final `state.book` exactly as the record builder snapshots engine state: `bids`, `asks`, `trades`, `iceberg_meta[:n_active]`, `n_active`, `ts_counter`, `book_full`, `truncated`; and (b) the 0-based index of the first step whose returned `done` flag is true, or −1 if none.
   - `jit=True` uses the environment's own jitted `reset` / `step_env`; `jit=False` runs the whole call under `jax.disable_jit()`. The function may raise.

Section 7 calls `run_record(..., jit=True)` on every contract of every draw and `run_record(..., jit=False)` on the contracts of the first `EAGER_DRAWS` valid draws. It reads `final_snapshot`, `observed_capacity` and the whole record through `run_battery` and `compare_records`. It calls `run_stacked(..., jit=True)` once per pooled stack of `STACK_BATCH` valid draws, once per draw-level stack and once per contract alone (`tapes = tape[None]`) on the first `ENV_ALONE_DRAWS` valid draws, and `run_stacked(..., jit=False)` on the draw-level stacks of the first `ENV_EAGER_DRAWS` valid draws. It reads the snapshot through `trades`, `resting`, `hidden_qty`, `book_full`, `truncated`, `ts_counter`, `n_active`, and, for eager vs jitted, every array of the snapshot.


In [ ]:
# ---------------------------------------------------------------------------------------
# Section 6 (K3): run_record and run_stacked, per the contract above.
#
# run_record: exactly the battery's record builder on one contract's tape, with the tape's
# analytic capacity bounds and no other argument (fresh OPEN book keyed by the tape's
# contract id, default product / market configurations, no cancels, no gate expiry).
#
# run_stacked: ONE Session built from {"tapes", "orders_per_step", "tape_seconds_span": 2.0 * M}
# only, M = float(max(int(tapes[..., C_TS].max()), 1)) over THIS call's own tapes (each call
# computes its own M; none is shared). Every other key at its default: default configurations,
# session_clock "span_scaled", and the default gct_minutes_from_got (not passed). With the
# span S' = 2M every slice clock stays at or below round(gct/2), so no lane reaches done within
# env.steps_per_episode steps (REL-10 repeat, K7-U verdict 2026-10-27; findings F8, F10). Every lane resets with
# its own key (jax.random.split(ENV_RESET_KEY, n)[i]), has the returned state's tape_idx set
# to its lane index i (finding F10), then plays exactly env.steps_per_episode calls of
# step_env with AgentOrder.none() (no participant order), called directly,
# never stopping at done. Every lane indexes the WHOLE stacked tape set held by that one
# environment instance. The done index is the first step whose returned done flag is true
# for that lane, or -1. Each lane's final book is snapshotted exactly as the record builder
# snapshots engine state (omie_invariants._snapshot on the lane's state.book).
#
#   jit=True: the environment's own @jax.jit reset / step_env under jax.vmap (training's
#     NUM_ENVS path), the vmapped functions jitted once per call. K3 DEVIATION (SCOPE-CHANGE
#     candidate, see the K3 report): the lanes are vmapped VMAP_WIDTH at a time (lanes
#     lo .. lo+VMAP_WIDTH-1 concurrently, then the next block, the last block possibly
#     narrower), not all n at once. Measured on this CPU, a vmapped step costs ~60 ms per
#     lane at width 1-4 but ~230-280 ms per lane at width 16-64 (under vmap the engine's
#     lax.cond early-outs become selects, so every bounded match loop runs to its full
#     trip count for every lane), so one all-lanes vmap of a 125-draw pooled stack
#     (~560 lanes) did not finish in the 7 200 s budget. Lanes in a block still run
#     concurrently through the same vmapped, jitted step over the same stacked tape set.
#   jit=False: the whole call runs under jax.disable_jit(). K3 DEVIATION (SCOPE-CHANGE
#     candidate): the lanes run ONE AT A TIME, unbatched, through the same environment
#     instance. An eager vmapped step did not complete in 400 s even at width 1 (the
#     batched match loop runs op by op for its full trip count), so the contract's eager
#     vmap cannot run; the unbatched eager lane is compared with the vmapped jitted lane.
#
#   Both modes run under an OUTER jax.experimental.enable_x64(), as a vmapped NUM_ENVS
#     rollout is wrapped: the
#     engine's int64 extract_fills accumulator does not lower under vmap otherwise.
# Nothing here alters a tape: contract ids, order ids, prices and timestamps are passed as
# built (finding F13).
# ---------------------------------------------------------------------------------------
import contextlib

import jax
import jax.experimental
import jax.numpy as jnp

from omiecid_simulator.omiecid.omie_session import AgentOrder, Session
from tests.omiecid.reference import omie_invariants as _inv
from tests.omiecid.reference.omie_invariants import build_record

ENV_RESET_KEY = jax.random.PRNGKey(0)
ENV_STEP_KEY = jax.random.PRNGKey(1)
VMAP_WIDTH = 4             # lanes per vmapped block on the jitted path (measured optimum, see above)
# Descriptive only: section 7 prints HORIZON in its F8 label. No gct_minutes_from_got is passed.
HORIZON = "the full-replay span tape_seconds_span = 2·M per Session (default gct)"


def run_record(tape, *, jit):
    """The isolated engine run: the record builder on this contract's tape alone."""
    return build_record(tape, capacity_bounds=synth.tape_capacity_bounds(tape), jit=jit)


def _lane(tree, i):
    return jax.tree_util.tree_map(lambda x: x[i], tree)


def _run_block_vmapped(env, v_reset, v_step, keys, lo, hi):
    """Lanes lo..hi-1 concurrently under the jitted vmap; returns [(state.book, first_done)]."""
    w = hi - lo
    state = v_reset(keys[lo:hi])
    state = state.replace(tape_idx=jnp.arange(lo, hi, dtype=jnp.int32))     # F10: lane i plays tape i
    noop = jax.tree_util.tree_map(lambda x: jnp.broadcast_to(x, (w,)), AgentOrder.none())
    first_done = np.full(w, -1, dtype=np.int64)
    for k in range(int(env.steps_per_episode)):
        state, _report, done = v_step(ENV_STEP_KEY, state, noop)
        d = np.asarray(done).reshape(w).astype(bool)
        first_done[(first_done < 0) & d] = k
    if not np.array_equal(np.asarray(state.tape_idx), np.arange(lo, hi)):
        raise ValueError("run_stacked: a lane's tape_idx moved during the episode")
    return [(_lane(state.book, j), int(first_done[j])) for j in range(w)]


def _run_lane_unbatched(env, keys, i):
    """Lane i alone, unbatched (the eager path); returns (state.book, first_done)."""
    state = env.reset(keys[i])
    state = state.replace(tape_idx=jnp.int32(i))
    noop = AgentOrder.none()
    first_done = -1
    for k in range(int(env.steps_per_episode)):
        state, _report, done = env.step_env(ENV_STEP_KEY, state, noop)
        if first_done < 0 and bool(np.asarray(done)):
            first_done = k
    if int(state.tape_idx) != i:
        raise ValueError("run_stacked: the lane's tape_idx moved during the episode")
    return state.book, first_done


def run_stacked(tapes, *, orders_per_step, jit):
    """The environment's multi-contract path over one stacked tape set, lane i playing tape i."""
    tapes = np.asarray(tapes)
    if tapes.ndim != 3 or tapes.shape[-1] != synth.N_COLS or tapes.dtype != np.int32:
        raise ValueError(f"run_stacked: tapes must be an (n, N_padded, {synth.N_COLS}) int32 array, "
                         f"got {tapes.shape} {tapes.dtype}")
    ops = int(orders_per_step)
    if not 0 < ops <= PAD_EXTRA:
        raise ValueError(f"run_stacked: orders_per_step {ops} outside [1, {PAD_EXTRA}] (finding F9)")
    n = int(tapes.shape[0])
    span_max_ts = float(max(int(tapes[..., C_TS].max()), 1))     # M over this call's own tape set
    env = Session({"tapes": tapes, "orders_per_step": ops, "tape_seconds_span": 2.0 * span_max_ts})
    if int(env.n_tapes) != n:
        raise ValueError(f"run_stacked: env holds {env.n_tapes} tapes, expected {n}")
    keys = jax.random.split(ENV_RESET_KEY, n)
    out = []
    with jax.experimental.enable_x64():
        if jit:
            v_reset = jax.jit(jax.vmap(env.reset))
            v_step = jax.jit(jax.vmap(env.step_env, in_axes=(None, 0, 0)))
            for lo in range(0, n, VMAP_WIDTH):
                out.extend(_run_block_vmapped(env, v_reset, v_step, keys, lo, min(lo + VMAP_WIDTH, n)))
        else:
            with jax.disable_jit():
                for i in range(n):
                    out.append(_run_lane_unbatched(env, keys, i))
        return tuple((_inv._snapshot(book), done) for book, done in out)


print(f"section 6 wired: run_record (build_record on the contract's own tape) and run_stacked (one Session "
      f"over the stacked tape set, tape_idx pinned per lane, AgentOrder.none() step_env; jitted lanes vmapped "
      f"{VMAP_WIDTH} at a time, eager lanes unbatched)")

## 7. Certification

Three phases. No check stops the notebook: every assertion is evaluated and recorded, and a raise inside a draw or a stacked run is a failed check.

1. **Draws and isolated runs.** Per slot: build the draw (section 4), check the precondition **P**, run every contract through `run_record` jitted, fold the contracts' batteries into one report for the draw (worst status per check), and keep the isolated outcome. A generation breach, a failed precondition, a run that raised or an I8 `invalid` **discards** the slot, which is re-drawn from its replacement seeds. A failed precondition or a raise is also recorded as a **failed** check first, so it never passes silently. On the first `EAGER_DRAWS` valid draws, every contract also runs eager, and the record must equal the jitted one (`compare_records`).
2. **Pooled stacks (S, S′, V).** The valid draws, `STACK_BATCH` at a time, with all their contracts shuffled into one stack, run once through `run_stacked`. Every lane's outcome must equal its isolated outcome.
3. **Draw-level stacks and the environment alone.** On the first `ENV_ALONE_DRAWS` valid draws: the draw's own stack (distinct ids, draw order), and every contract alone in the environment. Each must equal the isolated outcome. On the first `ENV_EAGER_DRAWS`, the draw-level stack also runs eager, and every array of every final book must equal the jitted run.

**What is recorded** (compact, in full; finding F14): one oracle assertion per pooled stack whose `observed` holds one string per draw, `seed|digest12|ops<k>|<cid>:<orders>:<trades>:<digest10>:<S><A>;…|R<RB…I9>|E<eager>`. Here `S` is the pooled-stack letter and `A` the alone-environment letter (`p` pass, `f` fail, `x` the run raised, `-` not run). `R` gives one battery letter per check (`p`, `f`, `n` not applicable, `i` invalid). `E` is `+`/`-` for an eager comparison and `.` for none. Also one assertion per failing contract with the full difference, the battery folded per draw, and the aggregate assertions of section 1.

**Reported, not asserted:** trade timestamps and the sequence counter, stacked vs isolated (F6); first `done` step per lane, draw-level stack vs alone, measured under the full-replay span, expected −1 for every lane (F8); per-contract euro value; capacity peaks; discard rate; phase timings.


In [ ]:
import time as _time
import traceback

from tests.omiecid.reference import omie_invariants as inv

cert = Certification(
    cert_id=CERT_ID, notebook=NOTEBOOK, scenario="Multi-contract isolation",
    known_result="for every contract of every draw, its lane in a stacked multi-contract environment run ends with "
                 "exactly the outcome of its isolated engine run (trade rows except timestamps, resting book per "
                 "side, hidden remainder per iceberg, book_full/truncated); no lane trades a foreign order id; the "
                 "integer trade value W_c is identical (d = 0.25 h); the contract alone in the environment agrees "
                 "with both; the isolated runs pass the battery",
    oracle_kind="exact-parametric",
    quotes=QUOTES, citations=CITATIONS, constructions=(CONSTRUCTION,), requested_draws=int(DRAWS),
    runtime_forecast_s=240.0, started_at=T0,
    downstream="every multi-contract use of Session (training over a tape set with NUM_ENVS lanes, the per-tape "
               "evaluation driver, build_fixed_tape_set consumers); any future multi-contract episode (Stage 4+)",
)

R_WIRE = "independence rule 2: K3 wires section 6"
R_CEIL = "the capacity-validity precondition: 100 per side, 100 trades, 200 iterations, 1024 icebergs, 64 slices"
R_DATA = "dataset precondition (section 4): §2.1, §2.3, §3.1, §13.6 item 1, slice margin, envelope, overlap, stagger"
R_RUN = "the run completes (a raise is a failure)"
R_ISO = "§2.2 (own independent order book), §6.4 (never matched against each other): the strong form"
R_WEAK = "§6.4, §6.2 item 2 (same contract): no lane trades a foreign order id"
R_VAL = "§8.1, §2.3: V = Q x p x d, d = 0.25 h, a contract property"
R_96 = "§2.1, §2.2: 96 quarter-hourly contracts"
R_SLICE = "Session per-step slicing: the final clamped slice must read pad rows only (finding F9)"
R_COVER = "C16 coverage (sections 1 and 3)"

CHECK_IDS = ("RB", "I1", "I2", "I3", "I4", "I5", "I6", "I7", "I8", "I9")
CODE = {"pass": "p", "fail": "f", "not_applicable": "n", "invalid": "i"}

# ---- wiring and configuration ----------------------------------------------------------------------------------
WIRED = callable(globals().get("run_record")) and callable(globals().get("run_stacked"))
cert.check("section 6 wired: run_record and run_stacked are defined", WIRED, expected=True, observed=WIRED,
           unit="bool", rule=R_WIRE, detail="" if WIRED else "run_record / run_stacked undefined: no draw is run")
cert.check_equal("the five configured ceilings equal the documented ones", list(CEILINGS.values()),
                 DOCUMENTED_CEILINGS, unit="count", rule=R_CEIL)
cert.check_equal("§2.1: the market configuration models 96 contracts", NUM_CONTRACTS, 96, unit="count", rule=R_96)
cert.check("§8.1: d = 0.25 h, the duration of the 15-minute MTU", D_HOURS == 0.25 and int(CONFIG.mtu_minutes) == 15
           and D_HOURS * 60 == int(CONFIG.mtu_minutes), expected={"delivery_duration_hours": 0.25, "mtu_minutes": 15},
           observed={"delivery_duration_hours": D_HOURS, "mtu_minutes": int(CONFIG.mtu_minutes)}, unit="other",
           rule=R_VAL)
cert.check("environment slicing precondition: every orders_per_step <= the trailing pad rows",
           all(0 < int(o) <= PAD_EXTRA for o in ORDERS_PER_STEP_CYCLE), expected=f"each in [1, {PAD_EXTRA}]",
           observed=list(ORDERS_PER_STEP_CYCLE), unit="count", rule=R_SLICE)


# ---- helpers ------------------------------------------------------------------------------------------------------
def combine(cid, results):
    st = [r.status for r in results]
    status = ("invalid" if "invalid" in st else "fail" if "fail" in st else "pass" if "pass" in st
              else "not_applicable")
    return inv.CheckResult(check_id=cid, status=status, violations=tuple(v for r in results for v in r.violations),
                           unobserved=tuple(sorted({u for r in results for u in r.unobserved})),
                           evidence={"per_contract": st})


def combine_reports(reports):
    by = [{r.check_id: r for r in rep.results} for rep in reports]
    return inv.BatteryReport(results=tuple(combine(cid, [b[cid] for b in by]) for cid in CHECK_IDS))


def snapshot_diff(a, b):
    """Names of the snapshot fields that differ, every array included (eager vs jitted)."""
    out = []
    for name in ("bids", "asks", "trades", "iceberg_meta"):
        x, y = np.asarray(getattr(a, name)), np.asarray(getattr(b, name))
        if x.shape != y.shape or not np.array_equal(x, y):
            out.append(name)
    for name in ("n_active", "ts_counter", "book_full", "truncated"):
        if getattr(a, name) != getattr(b, name):
            out.append(name)
    return out


def tb():
    return traceback.format_exc(limit=6)[-1500:]


# ---- phase 1: draws and isolated engine runs ---------------------------------------------------------------------
VALID = []                 # one dict per valid draw, in slot order
PRECOND_FAILS = 0
TIMING = {}
_t = _time.time()
if WIRED:
    for j in range(int(DRAWS)):
        for r in range(int(REPLACEMENTS_PER_SLOT) + 1):
            seed = slot_seed(j, r)
            try:
                draw = make_draw(seed)
            except (synth.CapacityBreach, DrawBuildFailure) as exc:
                cert.discard(seed, f"generation: {type(exc).__name__}: {exc}")
                continue
            probs = precondition(draw)
            if probs:
                PRECOND_FAILS += 1
                cert.check(f"seed {seed}: dataset precondition", False, seed=seed, expected="no problem",
                           observed=probs, unit="other", rule=R_DATA)
                cert.discard(seed, "dataset precondition failed")
                continue
            try:
                recs = [run_record(c.tape, jit=True) for c in draw.contracts]
            except Exception:  # noqa: BLE001
                cert.check(f"seed {seed}: isolated engine run raised", False, seed=seed, expected="a RunRecord",
                           observed=tb(), unit="other", rule=R_RUN)
                cert.discard(seed, "isolated run raised")
                continue
            reports = [inv.run_battery(rec, expects_trade=implies_trade(c.tape))
                       for c, rec in zip(draw.contracts, recs)]
            combined = combine_reports(reports)
            if not cert.add_battery(combined, seed=seed):
                continue                       # I8 invalid: discarded and recorded by the harness
            for rec in recs:
                cert.record_capacity({**rec.observed_capacity, "hops": None})   # hops: unobserved by the engine
            cert.record_case(seed, draw.combined, draw.params)
            eager = None
            if len(VALID) < int(EAGER_DRAWS):
                diffs = {}
                try:
                    for c, rec in zip(draw.contracts, recs):
                        d = inv.compare_records(run_record(c.tape, jit=False), rec)
                        if d:
                            diffs[c.cid] = list(d)
                    eager = not diffs
                    cert.set_eager_jit(seed, eager, detail="; ".join(f"contract {k}: {', '.join(v)}"
                                                                   for k, v in diffs.items()))
                except Exception:  # noqa: BLE001
                    eager = False
                    cert.set_eager_jit(seed, False, detail="eager record run raised: " + tb())
            iso = [outcome_of(rec.final_snapshot, c.ice_oids) for c, rec in zip(draw.contracts, recs)]
            VALID.append({
                "seed": seed, "draw": draw, "iso": [o for o, _a in iso], "iso_aux": [a for _o, a in iso],
                "letters": "".join(CODE[combined.statuses()[cid]] for cid in CHECK_IDS),
                "eager": eager, "S": ["-"] * len(draw.contracts), "A": ["-"] * len(draw.contracts), "ops": None,
            })
            del recs
            break
TIMING["phase 1 (draws, isolated engine runs)"] = round(_time.time() - _t, 1)

OID_OWNER = {}             # seed -> {order id: contract id} (S' diagnosis)
for v in VALID:
    OID_OWNER[v["seed"]] = {int(o): c.cid for c in v["draw"].contracts for o in real_rows(c.tape)[:, C_OID]}
W_MISMATCH = []
AUX = {"stamps_differ": 0, "ts_counter_differs": 0, "lanes": 0}
V_EUR = []


def judge_lane(v, ci, snap, where, run_label):
    """Compare one lane's final book with the contract's isolated outcome; record a failure in full."""
    c = v["draw"].contracts[ci]
    exp, exp_aux = v["iso"][ci], v["iso_aux"][ci]
    obs, obs_aux = outcome_of(snap, c.ice_oids)
    diff = compare_outcomes(exp, obs)
    own = {int(o) for o in real_rows(c.tape)[:, C_OID]}
    foreign = foreign_oids(obs, own, OID_OWNER.get(v["seed"], {}))
    w_exp, w_obs = trade_value_w(exp), trade_value_w(obs)
    if w_exp != w_obs:
        W_MISMATCH.append([v["seed"], c.cid, run_label, w_exp, w_obs])
    AUX["lanes"] += 1
    AUX["stamps_differ"] += int(exp_aux["stamps"] != obs_aux["stamps"])
    AUX["ts_counter_differs"] += int(exp_aux["ts_counter"] != obs_aux["ts_counter"])
    ok = not diff and not foreign
    if not ok:
        cert.check(f"seed {v['seed']}: contract {c.cid} {run_label} == its isolated engine run", False, seed=v["seed"],
                   expected={"trade_digest": trade_digest(exp), "n_trades": len(exp["trades"])},
                   observed={"trade_digest": trade_digest(obs), "n_trades": len(obs["trades"]), "diff": diff,
                             "foreign_oids": foreign, "where": where, "trade_columns": list(TRADE_KEY_NAMES)},
                   unit="other", rule=R_ISO if diff else R_WEAK)
    return ok, obs


# ---- phase 2: pooled stacks (S, S', V) ------------------------------------------------------------------------------
_t = _time.time()
N_STACKED_DRAWS = 0
STACK_LOG = []
if WIRED and VALID:
    for b, lo in enumerate(range(0, len(VALID), int(STACK_BATCH))):
        batch = list(range(lo, min(lo + int(STACK_BATCH), len(VALID))))
        ops = int(ORDERS_PER_STEP_CYCLE[b % len(ORDERS_PER_STEP_CYCLE)])
        lanes = [(vi, ci) for vi in batch for ci in range(len(VALID[vi]["draw"].contracts))]
        perm = np.random.Generator(np.random.PCG64(int(SEED0) + 7919 * (b + 1))).permutation(len(lanes))
        lanes = [lanes[int(p)] for p in perm]
        stack = synth.stack_tapes([VALID[vi]["draw"].contracts[ci].tape for vi, ci in lanes], pad_extra=PAD_EXTRA)
        for vi in batch:
            VALID[vi]["ops"] = ops
        try:
            out = run_stacked(stack, orders_per_step=ops, jit=True)
            if len(out) != len(lanes):
                raise ValueError(f"run_stacked returned {len(out)} lanes for {len(lanes)} tapes")
        except Exception:  # noqa: BLE001
            cert.check(f"pooled stack {b}: the stacked run completes", False, expected=f"{len(lanes)} lanes",
                       observed=tb(), unit="other", rule=R_RUN)
            for vi, ci in lanes:
                VALID[vi]["S"][ci] = "x"
            continue
        for k, (vi, ci) in enumerate(lanes):
            snap, _done = out[k]
            ok, obs = judge_lane(VALID[vi], ci, snap, {"pooled_stack": b, "lane": k, "n_lanes": len(lanes),
                                                       "orders_per_step": ops, "stack_shape": list(stack.shape)},
                                 "in a pooled stack")
            VALID[vi]["S"][ci] = "p" if ok else "f"
            V_EUR.append(value_eur(trade_value_w(obs)))
        N_STACKED_DRAWS += len(batch)
        STACK_LOG.append({"stack": b, "draws": len(batch), "lanes": len(lanes), "orders_per_step": ops,
                          "shape": list(stack.shape)})
TIMING["phase 2 (pooled stacks)"] = round(_time.time() - _t, 1)

# ---- phase 3: draw-level stacks, the environment alone, environment eager -------------------------------------------
_t = _time.time()
N_ALONE = 0
N_ENV_EAGER = 0
DONE_REPORT = []
if WIRED:
    for vi in range(min(int(ENV_ALONE_DRAWS), len(VALID))):
        v = VALID[vi]
        cs = v["draw"].contracts
        ops = int(ORDERS_PER_STEP_CYCLE[vi % len(ORDERS_PER_STEP_CYCLE)])
        own_stack = synth.stack_tapes([c.tape for c in cs], pad_extra=PAD_EXTRA)
        own_out = None
        try:
            own_out = run_stacked(own_stack, orders_per_step=ops, jit=True)
            if len(own_out) != len(cs):
                raise ValueError(f"run_stacked returned {len(own_out)} lanes for {len(cs)} tapes")
            letters = []
            for ci in range(len(cs)):
                ok, _obs = judge_lane(v, ci, own_out[ci][0], {"draw_stack": True, "lane": ci, "orders_per_step": ops},
                                      "in its draw's own stack")
                letters.append("p" if ok else "f")
            cert.check(f"seed {v['seed']}: the draw's own stack ({len(cs)} distinct ids) equals the isolated runs",
                       all(x == "p" for x in letters), seed=v["seed"], expected="p" * len(cs),
                       observed="".join(letters), unit="other", rule=R_ISO)
        except Exception:  # noqa: BLE001
            own_out = None
            cert.check(f"seed {v['seed']}: the draw's own stacked run completes", False, seed=v["seed"],
                       expected=f"{len(cs)} lanes", observed=tb(), unit="other", rule=R_RUN)
        for ci, c in enumerate(cs):
            try:
                alone = run_stacked(c.tape[None], orders_per_step=ops, jit=True)
                ok, obs = judge_lane(v, ci, alone[0][0], {"alone": True, "orders_per_step": ops},
                                     "alone in the environment")
                if own_out is not None:
                    lane_obs, _ = outcome_of(own_out[ci][0], c.ice_oids)
                    d = compare_outcomes(lane_obs, obs)
                    if d:
                        ok = False
                        cert.check(f"seed {v['seed']}: contract {c.cid} alone in the environment == its lane in the "
                                   f"draw's stack", False, seed=v["seed"], expected="identical", observed=d,
                                   unit="other", rule=R_ISO)
                    DONE_REPORT.append({"seed": v["seed"], "cid": c.cid, "done_step_in_stack": int(own_out[ci][1]),
                                        "done_step_alone": int(alone[0][1])})
                v["A"][ci] = "p" if ok else "f"
            except Exception:  # noqa: BLE001
                v["A"][ci] = "x"
                cert.check(f"seed {v['seed']}: contract {c.cid} alone in the environment completes", False,
                           seed=v["seed"], expected="a lane", observed=tb(), unit="other", rule=R_RUN)
            N_ALONE += 1
        if vi < int(ENV_EAGER_DRAWS):
            try:
                eager_out = run_stacked(own_stack, orders_per_step=ops, jit=False)
                if own_out is None:
                    raise ValueError("no jitted draw-level run to compare with")
                diffs = {cs[ci].cid: snapshot_diff(eager_out[ci][0], own_out[ci][0]) for ci in range(len(cs))}
                diffs = {k: d for k, d in diffs.items() if d}
                dones = [[int(eager_out[ci][1]), int(own_out[ci][1])] for ci in range(len(cs))
                         if int(eager_out[ci][1]) != int(own_out[ci][1])]
                ok = not diffs and not dones
                cert.set_eager_jit(v["seed"], ok, detail="environment stack: " + (
                    "; ".join(f"contract {k}: {', '.join(d)}" for k, d in diffs.items())
                    + (f"; done steps differ {dones}" if dones else "") if not ok else "identical"))
            except Exception:  # noqa: BLE001
                cert.set_eager_jit(v["seed"], False, detail="environment stack eager run failed: " + tb())
            N_ENV_EAGER += 1
TIMING["phase 3 (draw-level stacks, alone, eager)"] = round(_time.time() - _t, 1)

# ---- per-stack records (compact, one string per draw) --------------------------------------------------------------
def draw_string(v):
    parts = []
    for ci, c in enumerate(v["draw"].contracts):
        parts.append(f"{c.cid}:{c.n_orders}:{len(v['iso'][ci]['trades'])}:{trade_digest(v['iso'][ci])}:"
                     f"{v['S'][ci]}{v['A'][ci]}")
    e = "." if v["eager"] is None else ("+" if v["eager"] else "-")
    return f"{v['seed']}|{v['draw'].digest[:12]}|ops{v['ops']}|{';'.join(parts)}|R{v['letters']}|E{e}"


for b, lo in enumerate(range(0, len(VALID), int(STACK_BATCH))):
    vs = VALID[lo:lo + int(STACK_BATCH)]
    ok = all(all(x == "p" for x in v["S"]) and all(x in "p-" for x in v["A"]) for v in vs)
    cert.check(f"pooled stack {b}: every contract of draws {vs[0]['seed']}-{vs[-1]['seed']} equals its isolated run",
               ok, expected="S = p for every contract; A in {p, -}", observed=[draw_string(v) for v in vs],
               unit="other", rule=R_ISO)

# ---- aggregate assertions -------------------------------------------------------------------------------------------
_req = int(DRAWS)
cert.check_equal("dataset precondition: failed draws", PRECOND_FAILS, 0, unit="count", rule=R_DATA)
cert.check_equal("every valid draw ran in a pooled stacked run", N_STACKED_DRAWS, len(VALID), unit="count",
                 rule=R_COVER)
cert.check_equal("§8.1: per-contract integer trade value W_c identical in every arm (mismatches)",
                 W_MISMATCH, [], unit="other", rule=R_VAL)
cert.check_equal("record eager comparisons made (absence is a failure)",
                 sum(v["eager"] is not None for v in VALID), min(int(EAGER_DRAWS), _req), unit="count",
                 rule="eager == jitted on valid draws (harness lesson a)")
cert.check_equal("environment eager comparisons made (absence is a failure)", N_ENV_EAGER,
                 min(int(ENV_EAGER_DRAWS), _req), unit="count", rule="eager == jitted on valid draws (harness lesson a)")
_alone_expected = sum(len(v["draw"].contracts) for v in VALID[:min(int(ENV_ALONE_DRAWS), _req)])
cert.check("environment-alone comparisons made", N_ALONE == _alone_expected and N_ALONE >= 3 * min(int(ENV_ALONE_DRAWS), _req),
           expected=f"{_alone_expected} (>= 3 per draw)", observed=N_ALONE, unit="count", rule=R_COVER)
_ids = sorted({c.cid for v in VALID for c in v["draw"].contracts})
cert.check("§2.1: contract ids 0 and 95 (both ends of the day) are exercised", {0, NUM_CONTRACTS - 1} <= set(_ids),
           expected=[0, NUM_CONTRACTS - 1], observed={"distinct_ids": len(_ids), "min": _ids[0] if _ids else None,
                                                      "max": _ids[-1] if _ids else None}, unit="other", rule=R_96)

display(Markdown(cert.table()))

# ---- reported, never asserted -----------------------------------------------------------------------------------------
n_contracts = sum(len(v["draw"].contracts) for v in VALID)
n_trading = sum(1 for v in VALID for o in v["iso"] if o["trades"])
print(f"valid draws {len(VALID)}; contracts {n_contracts} ({n_trading} with at least one trade); "
      f"distinct ids {len(_ids)}; pooled stacks {STACK_LOG}")
print(f"F6, reported: of {AUX['lanes']} lane comparisons, trade timestamps differ from the isolated run in "
      f"{AUX['stamps_differ']}, the sequence counter in {AUX['ts_counter_differs']}")
print(f"F8, reported (measured under the explicit horizon gct_minutes_from_got = {HORIZON}): first done step, "
      f"draw-level stack vs alone: {DONE_REPORT}")
if V_EUR:
    print(f"per-contract trade value V_c (EUR, stacked lanes): min {min(V_EUR):.2f}, max {max(V_EUR):.2f}, "
          f"sum {sum(V_EUR):.2f}")
print(f"timings (s): {TIMING}")


## 8. Verdict

Emit the results JSON (validated against `results.schema.json`) and print exactly one word:
`PASS` or `NOT CERTIFIED`. On `NOT CERTIFIED` the smallest failing case is frozen as a fixture.


In [ ]:
RESULTS_PATH = cert.emit()
print(cert.verdict_line())
print(f"results -> {RESULTS_PATH}")

In [ ]:
# Section 9 ("Why this certification failed") is rendered here only on NOT CERTIFIED; it is
# persisted as the last markdown cell at K6-F by cert_support.append_failure_section.
_section9 = cert.failure_section_markdown()
if _section9:
    display(Markdown(_section9))

## 9. Why this certification failed

### 9.1 What failed

3 assertion(s) failed, first 'seed 235: contract 95 in a pooled stack == its isolated engine run' on seed 235: expected {'trade_digest': 'd39cd7d6cd', 'n_trades': 5}, observed {'trade_digest': '88deba6710', 'n_trades': 4, 'diff': {'trades': {'n_expected': 5, 'n_observed': 4, 'rows': [[4, [1005282, 39, 12394006, 12394032, 1, 1], None]]}, 'bids': {'missing': [[1005282, 111, 12394006, 1]], 'extra': [[1005282, 150, 12394006, 1]]}}, 'foreign_oids': {}, 'where': {'pooled_stack': 1, 'lane': 583, 'n_lanes': 584, 'orders_per_step': 3, 'stack_shape': [584, 64, 13]}, 'trade_columns': ['price_int', 'qty_signed', 'passive_oid', 'aggressor_oid', 'passive_tid', 'aggressor_tid']} (other).

### 9.2 The smallest failing case

Seed 235, 152 real tape rows; frozen fixture `notebooks/certification/fixtures/C16_seed235.npz` (keys `tape`, `params_json`).

### 9.3 Which rule it concerns

§2.2 (own independent order book), §6.4 (never matched against each other): the strong form

### 9.4 Where the fault most plausibly lies

Hypothesis: The notebook's stacked-run configuration, not matching or isolation. The failing lane (contract 95 of seed 235, the last lane of pooled stack 1) holds the stack's largest timestamp, 110395 s, and its second-to-last slice of three orders (rows 27-29) already carries 110395. Under the session's default span_scaled clock a slice's time is round(m_k * gct / S), with S the stack's largest timestamp, so that slice reaches the gate-closure time and ends the session; the session ignores every later call, and the last slice (rows 30-31, the 39-tick sell that trades against the resting bid of 185) is never replayed: exactly the missing fifth trade and the bid resting at 150 instead of 111. Setting gct_minutes_from_got to 1,000,000 does not prevent this, because the span_scaled clock scales with gct; a tape_seconds_span beyond every lane's largest timestamp would. The same draw passed in the earlier sealed run, whose stacked environment did not stop replaying at done.

### 9.5 What it affects downstream

any multi-contract Session run under the span_scaled clock whose last slices share the stack's largest timestamp; C16 results for the other 499 draws are unaffected
